In [ ]:
# =========================================
# Stroke Prediction System (Full Working)
# =========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer

from imblearn.over_sampling import RandomOverSampler

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report

# ========= 1. Load Data =========
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# ========= 2. Missing Value Handling =========
imputer = KNNImputer(n_neighbors=5)
df['bmi'] = imputer.fit_transform(df[['bmi']])

# ========= 3. Outlier Removal =========
for col in ['avg_glucose_level', 'bmi']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

# ========= 4. Encoding =========
categorical_cols = ['gender','ever_married','work_type','Residence_type','smoking_status']

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# ========= 5. Features & Target =========
X = df.drop(['id','stroke'], axis=1)
y = df['stroke']

# ========= 6. Scaling =========
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ========= 7. Train Test Split =========
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# ========= 8. Balance Data =========
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# ========= 9. Models =========
rf = RandomForestClassifier(n_estimators=100, random_state=42)
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(eval_metric='logloss', random_state=42)

model = VotingClassifier(
    estimators=[('rf', rf), ('et', et), ('xgb', xgb)],
    voting='soft'
)

# ========= 10. Train =========
model.fit(X_train_bal, y_train_bal)

# ========= 11. Evaluation =========
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

# =========================================
# 🔥 12. MANUAL INPUT PREDICTION SYSTEM
# =========================================

print("\n===== Stroke Prediction Input =====")

age = float(input("Age: "))
hypertension = int(input("Hypertension (0/1): "))
heart_disease = int(input("Heart Disease (0/1): "))
avg_glucose_level = float(input("Avg Glucose Level: "))
bmi = float(input("BMI: "))

gender = input("Gender: ")
ever_married = input("Ever Married: ")
work_type = input("Work Type: ")
Residence_type = input("Residence Type: ")
smoking_status = input("Smoking Status: ")

# create input dataframe
input_data = pd.DataFrame([[
    age, hypertension, heart_disease,
    avg_glucose_level, bmi,
    gender, ever_married,
    work_type, Residence_type,
    smoking_status
]], columns=X.columns)

# encode categorical
for col in categorical_cols:
    input_data[col] = encoders[col].transform(input_data[col])

# scale
input_scaled = scaler.transform(input_data)

# predict
result = model.predict(input_scaled)

print("\n========================")

if result[0] == 1:
    print("⚠️ HIGH STROKE RISK")
else:
    print("✅ LOW / NO STROKE RISK")

print("========================")


Accuracy: 0.9612314709236032
Recall: 0.06060606060606061
F1 Score: 0.10526315789473684

Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       844
           1       0.40      0.06      0.11        33

    accuracy                           0.96       877
   macro avg       0.68      0.53      0.54       877
weighted avg       0.94      0.96      0.95       877


===== Stroke Prediction Input =====


Age:  67
Hypertension (0/1):  1
Heart Disease (0/1):  0
Avg Glucose Level:  180
BMI:  31
Gender:  Male
Ever Married:  Yes
